<a href="https://colab.research.google.com/github/LBJLincoln/nomos-nba-agent/blob/main/colab/nba_evolution_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Quant AI — Genetic Evolution with GPU (Google Colab)

**One-click notebook**: Clone repo → install deps → run genetic evolution on T4 GPU.

Uses the **full NBAFeatureEngine v3.0-35cat** (6000+ features) — same as S10/S11 HF Spaces.

**Instructions**:
1. Open in Colab: Runtime → Change runtime type → GPU (T4 free)
2. Add Colab Secrets (key icon left sidebar):
   - `DATABASE_URL`: your Supabase postgres URL
3. Run all cells — evolution starts automatically

## 1. Clone Repo & Install

In [5]:
import subprocess, sys, os

if not os.path.exists("/content/nomos-nba-agent"):
    subprocess.run(["git", "clone", "https://github.com/LBJLincoln/nomos-nba-agent.git",
                    "/content/nomos-nba-agent"], check=True)
else:
    subprocess.run(["git", "-C", "/content/nomos-nba-agent", "pull"], check=True)

os.chdir("/content/nomos-nba-agent")
sys.path.insert(0, "/content/nomos-nba-agent")

_pkgs = [
    "nba_api",  # Required for NBAFeatureEngine (6000+ features)
    "psycopg2-binary", "xgboost", "lightgbm", "catboost",
    "pytorch-tabnet", "scikit-learn>=1.3", "pandas", "numpy", "scipy",
]
for pkg in _pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

# Verify engine loads
try:
    from features.engine import NBAFeatureEngine
    print(f"NBAFeatureEngine: OK")
except Exception as e:
    print(f"WARNING: NBAFeatureEngine failed: {e} (will use 250-feature fallback)")

PyTorch 2.10.0+cu128 — CUDA: True
  GPU: Tesla T4 (15.6 GB)
NBAFeatureEngine: OK


## 2. Load Secrets

In [6]:
# Load DATABASE_URL BEFORE importing evolution (RunLogger checks at import time)
try:
    from google.colab import userdata
    _db_url = userdata.get("DATABASE_URL")
    if _db_url:
        os.environ["DATABASE_URL"] = _db_url
        print(f"DATABASE_URL loaded from Colab secrets ({_db_url[:30]}...)")
    else:
        print("WARNING: DATABASE_URL secret is empty")
except Exception:
    if not os.environ.get("DATABASE_URL"):
        print("Set DATABASE_URL manually: os.environ['DATABASE_URL'] = 'postgresql://...'")
    else:
        print(f"DATABASE_URL already set in env")

Set DATABASE_URL manually: os.environ['DATABASE_URL'] = 'postgresql://...'


## 3. Run Genetic Evolution (GPU)

- **Pop: 100** (5 islands × 20) — fast iterations on T4
- **10 gens/cycle** × 50 cycles = **500 generations total**
- All model types: XGBoost, LightGBM, CatBoost, TabNet, MLP, LSTM
- Walk-forward 3-fold, Supabase logging
- Uses real NBAFeatureEngine (6000+ features, same as S10/S11)

In [ ]:
from evolution.genetic_loop_v3 import run_continuous

# GPU evolution: 100 pop, 10 gens/cycle, 50 cycles
run_continuous(
    generations_per_cycle=10,
    total_cycles=50,
    pop_size=100,
    target_features=100,
    n_splits=3,
    cool_down=10,
)

  NBA QUANT AI — REAL GENETIC EVOLUTION LOOP v3
  Started: 2026-03-24T22:43:50.052452+00:00
  Pop: 100 | Target features: 100
  Gens/cycle: 10 | Cycles: 50

[PHASE 1] Loading data...
[DATA] All 8 seasons cached
  9551 games loaded

[PHASE 2] Building features...
[ENGINE] Real NBAFeatureEngine: 5961 features, 9457 games
  Feature matrix: (9457, 5961) (5961 features)

[PHASE 3] Initializing engine...
[GPU] XGBoost CUDA: ENABLED
[INIT] Population: 100 individuals, 5961 feature candidates, ~100 target features
[RUN-LOGGER] Supabase logging + auto-cut ACTIVE

  CYCLE 1 — Starting 10 generations
  Time: 2026-03-24 23:18:37 UTC


## 4. Check Results

In [ ]:
import json
from pathlib import Path

f = Path("/content/nomos-nba-agent/data/results/evolution-latest.json")
if f.exists():
    r = json.loads(f.read_text())
    b = r.get("best", {})
    print(f"Best Brier: {b.get('brier')} | ROI: {b.get('roi')} | "
          f"Sharpe: {b.get('sharpe')} | Features: {b.get('n_features')} | "
          f"Model: {b.get('model_type')} | Gen: {r.get('generation')}")
else:
    print("No results yet.")